<a href="https://colab.research.google.com/github/karkessler/dhbw-mathe3/blob/main/notebooks/numerik/06_newton_mehrdimensional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mehrdimensionales Newton-Verfahren: Gleichungssysteme und Optimierung

In den vorherigen Abschnitten haben wir das eindimensionale Newton-Verfahren zur Nullstellensuche (Notebook `00_nullstellenverfahren.ipynb`) sowie den Gradientenabstieg zur mehrdimensionalen Optimierung (Notebook `05_gradient_descent_rosenbrock.ipynb`) kennengelernt.

In diesem Notebook übertragen wir das Newton-Verfahren auf den mehrdimensionalen Raum $\mathbb{R}^n$:

1. **Newton-Verfahren für nichtlineare Gleichungssysteme $F(x) = 0$:**
   Hier ersetzt die **Jacobi-Matrix** $J_F$ die gewöhnliche Ableitung. Wir untersuchen das geometrische Problem aus dem Schnitt von Kreis und Hyperbel ($x^2 + y^2 = 4$ und $xy = 1$).
2. **Newton-Verfahren zur mehrdimensionalen Minimierung:**
   Hier wird das Verfahren auf die Gradientenbedingung $\nabla f(x) = 0$ angewendet. Die dabei benötigte Ableitung des Gradienten ist die **Hesse-Matrix** $H_f$. Wir demonstrieren die quadratische Konvergenz an der Rosenbrock-Funktion im direkten Vergleich zum Gradientenabstieg.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('default')
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'savefig.transparent': False,
    'axes.edgecolor': 'black',
    'axes.labelcolor': 'black',
    'axes.titlecolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black',
    'axes.grid': True,
    'grid.color': '0.8',
    'legend.facecolor': 'white',
    'legend.edgecolor': '0.7',
})

# Teil 1: Nichtlineare Gleichungssysteme $F(x) = 0$

## Mathematische Herleitung

Gesucht ist ein Vektor $x^* \in \mathbb{R}^n$ mit $F(x^*) = 0$ für eine differenzierbare Funktion $F: \mathbb{R}^n \to \mathbb{R}^n$.

Lineare Taylor-Näherung um den aktuellen Iterationspunkt $x_k$:

$$F(x_k + \Delta x) \approx F(x_k) + J_F(x_k)\,\Delta x \overset{!}{=} 0$$

wobei $J_F(x_k)$ die **Jacobi-Matrix** (Funktionalmatrix) an der Stelle $x_k$ ist:

$$J_F(x) = \begin{pmatrix} \frac{\partial F_1}{\partial x_1} & \cdots & \frac{\partial F_1}{\partial x_n} \\ \vdots & \ddots & \vdots \\ \frac{\partial F_n}{\partial x_1} & \cdots & \frac{\partial F_n}{\partial x_n} \end{pmatrix}$$

In jedem Schritt löst man das **lineare Gleichungssystem**:

$$J_F(x_k)\,\Delta x_k = -F(x_k)$$

und setzt

$$x_{k+1} = x_k + \Delta x_k.$$

*(Hinweis: In der Praxis invertiert man $J_F$ nicht explizit, sondern löst das lineare Gleichungssystem mit `np.linalg.solve`.)*

## Beispiel: Schnitt Kreis $\cap$ Hyperbel

Wir betrachten das nichtlineare Gleichungssystem:

$$\begin{aligned}
F_1(x, y) &= x^2 + y^2 - 4 = 0 \quad &&\text{(Kreis mit Radius } r=2\text{)} \\
F_2(x, y) &= x \cdot y - 1 = 0 \quad &&\text{(Hyperbel } y = 1/x\text{)}
\end{aligned}$$

Die zugehörige Jacobi-Matrix lautet:

$$J_F(x, y) = \begin{pmatrix} 2x & 2y \\ y & x \end{pmatrix}$$

### Analytische Lösung
Durch Einsetzen von $y = 1/x$ in die Kreisgleichung erhält man $x^2 + \frac{1}{x^2} = 4 \iff x^4 - 4x^2 + 1 = 0$.
Die Lösungen im 1. Quadranten sind:

$$x^* = \sqrt{2 + \sqrt{3}} \approx 1{,}9318516526, \qquad y^* = \frac{1}{x^*} = \sqrt{2 - \sqrt{3}} \approx 0{,}5176380902$$

In [ ]:
def F_system(x, y):
    """Nichtlineares Gleichungssystem: Kreis und Hyperbel."""
    return np.array([x**2 + y**2 - 4.0, x * y - 1.0])

def J_system(x, y):
    """Jacobi-Matrix des Systems."""
    return np.array([
        [2.0 * x, 2.0 * y],
        [y, x]
    ])

# Exakte Lösung im 1. Quadranten
x_exakt_1 = np.sqrt(2.0 + np.sqrt(3.0))
y_exakt_1 = 1.0 / x_exakt_1
punkt_exakt = np.array([x_exakt_1, y_exakt_1])

print(f"Exakter Schnittpunkt (1. Quadrant): x = {x_exakt_1:.10f}, y = {y_exakt_1:.10f}")

## Iteration und Konvergenztabelle

In [ ]:
def newton_system(F, J, x0, tol=1e-12, max_iter=20):
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    errors = []
    residuals = []
    
    for k in range(max_iter):
        F_val = F(*x)
        res_norm = np.linalg.norm(F_val)
        err = np.linalg.norm(x - punkt_exakt)
        residuals.append(res_norm)
        errors.append(err)
        
        if res_norm < tol:
            break
            
        J_val = J(*x)
        delta = np.linalg.solve(J_val, -F_val)
        x = x + delta
        history.append(x.copy())
        
    return x, np.array(history), np.array(residuals), np.array(errors)

x0_sys = np.array([2.0, 0.2])
x_opt_sys, hist_sys, res_sys, err_sys = newton_system(F_system, J_system, x0_sys)

print("Newton-Verfahren für nichtlineares Gleichungssystem (Kreis & Hyperbel)")
print(f"Startpunkt: x_0 = ({x0_sys[0]:.2f}, {x0_sys[1]:.2f})\n")
print(f"{'k':>2} | {'x_k':>14} | {'y_k':>14} | {'||F(x_k)||':>14} | {'Fehler ||x_k - x*||':>20}")
print("-" * 72)
for k in range(len(hist_sys)):
    xk, yk = hist_sys[k]
    rk = res_sys[k] if k < len(res_sys) else np.linalg.norm(F_system(xk, yk))
    ek = err_sys[k] if k < len(err_sys) else np.linalg.norm(hist_sys[k] - punkt_exakt)
    print(f"{k:2d} | {xk:14.10f} | {yk:14.10f} | {rk:14.4e} | {ek:20.4e}")

Die Tabelle zeigt eindrucksvoll die **quadratische Konvergenz**: Der Fehler und das Residuum fallen ab der Nähe der Nullstelle extrem schnell ab; die Anzahl korrekter Nachkommastellen verdoppelt sich in jedem Iterationsschritt.

## Visualisierung der Iterationsschritte im Phasenraum

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

# Kreis x^2 + y^2 = 4
phi = np.linspace(0, 2 * np.pi, 400)
ax.plot(2 * np.cos(phi), 2 * np.sin(phi), 'b-', linewidth=2, label=r'Kreis $x^2 + y^2 = 4$')

# Hyperbel xy = 1
xh_pos = np.linspace(0.3, 3.0, 300)
xh_neg = np.linspace(-3.0, -0.3, 300)
ax.plot(xh_pos, 1.0 / xh_pos, 'g-', linewidth=2, label=r'Hyperbel $xy = 1$')
ax.plot(xh_neg, 1.0 / xh_neg, 'g-', linewidth=2)

# Alle 4 Schnittpunkte
s3 = np.sqrt(3.0)
s_plus = np.sqrt(2.0 + s3)
s_minus = np.sqrt(2.0 - s3)
schnittpunkte = [
    (s_plus, s_minus), (s_minus, s_plus),
    (-s_plus, -s_minus), (-s_minus, -s_plus)
]
for idx, (sp_x, sp_y) in enumerate(schnittpunkte):
    lbl = 'Schnittpunkte (exakt)' if idx == 0 else ''
    ax.plot(sp_x, sp_y, 'ko', markersize=6, label=lbl)

# Iterationspfad
ax.plot(hist_sys[:, 0], hist_sys[:, 1], 'r.-', linewidth=1.5, markersize=8, label='Newton-Schritte')
ax.plot(x0_sys[0], x0_sys[1], 'rs', markersize=8, label=r'Startpunkt $x_0 = (2{,}0,\ 0{,}2)$')

for k, (px, py) in enumerate(hist_sys[:5]):
    offset = (0.08, 0.08) if k != 0 else (0.08, -0.15)
    ax.annotate(f'$x_{k}$', (px, py), xytext=(px + offset[0], py + offset[1]),
                fontsize=10, fontweight='bold', color='darkred')

ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.set_xlabel('$x$')
ax.set_ylabel('$y$')
ax.set_title(r'Newton-Verfahren für Kreis $\cap$ Hyperbel')
ax.legend(loc='lower left', fontsize=8.5)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Teil 2: Newton-Verfahren zur Minimierung (Optimierung)

## Mathematische Herleitung

Gesucht ist das Minimum einer zweimal stetig differenzierbaren Funktion $f: \mathbb{R}^n \to \mathbb{R}$.

Eine notwendige Bedingung für ein lokales Minimum ist das Verschwinden des Gradienten:

$$\nabla f(x^*) = 0$$

Wir wenden das Newton-Verfahren auf das nichtlineare Gleichungssystem $F(x) = \nabla f(x) = 0$ an. Die Jacobi-Matrix des Gradienten $\nabla f$ ist die **Hesse-Matrix** $H_f$:

$$H_f(x) = \begin{pmatrix} \frac{\partial^2 f}{\partial x_1^2} & \cdots & \frac{\partial^2 f}{\partial x_1\,\partial x_n} \\ \vdots & \ddots & \vdots \\ \frac{\partial^2 f}{\partial x_n\,\partial x_1} & \cdots & \frac{\partial^2 f}{\partial x_n^2} \end{pmatrix}$$

### Newton-Iterationsvorschrift zur Minimierung:

$$H_f(x_k)\,\Delta x_k = -\nabla f(x_k), \qquad x_{k+1} = x_k + \Delta x_k$$

Im Vergleich zum Gradientenabstieg ($x_{k+1} = x_k - \eta \nabla f(x_k)$):
* **Gradientenabstieg:** nutzt nur die Information 1. Ordnung (Steigung) und bewegt sich orthogonal zu den Höhenlinien. In schmalen, gekrümmten Tälern führt dies zu typischem Zick-Zack-Verhalten und sehr langsamer Konvergenz.
* **Newton-Verfahren:** nutzt die Information 2. Ordnung (Krümmung via Hesse-Matrix) und passt Schrittweite sowie Richtung automatisch an die lokale quadratische Approximation an.

## Anwendung auf die Rosenbrock-Funktion

Die Rosenbrock-Funktion (aus Notebook 05):

$$f(x, y) = (1 - x)^2 + 100\,(y - x^2)^2$$

### Gradient und Hesse-Matrix:

$$\nabla f(x, y) = \begin{pmatrix} -2(1 - x) - 400x(y - x^2) \\ 200(y - x^2) \end{pmatrix}$$

$$H_f(x, y) = \begin{pmatrix} 2 - 400y + 1200x^2 & -400x \\ -400x & 200 \end{pmatrix}$$

In [ ]:
def rosenbrock(x, y, a=1.0, b=100.0):
    return (a - x)**2 + b * (y - x**2)**2

def rosenbrock_grad(x, y, a=1.0, b=100.0):
    dfdx = -2 * (a - x) - 4 * b * x * (y - x**2)
    dfdy = 2 * b * (y - x**2)
    return np.array([dfdx, dfdy])

def rosenbrock_hess(x, y, a=1.0, b=100.0):
    d2fdx2 = 2 - 4 * b * y + 12 * b * x**2
    d2fdxdy = -4 * b * x
    d2fdy2 = 2 * b
    return np.array([[d2fdx2, d2fdxdy], [d2fdxdy, d2fdy2]])

## Direkter Vergleich: Gradientenabstieg vs. Newton-Verfahren

In [ ]:
def gradient_descent(grad_f, x0, eta=0.001, tol=1e-6, max_iter=20000):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    for k in range(max_iter):
        g = grad_f(*x)
        if np.linalg.norm(g) < tol:
            break
        x = x - eta * g
        path.append(x.copy())
    return x, np.array(path)

def newton_optimizer(grad_f, hess_f, x0, tol=1e-6, max_iter=50):
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    for k in range(max_iter):
        g = grad_f(*x)
        if np.linalg.norm(g) < tol:
            break
        H = hess_f(*x)
        delta = np.linalg.solve(H, -g)
        x = x + delta
        path.append(x.copy())
    return x, np.array(path)

x0_opt = np.array([-1.2, 1.0])

# 1. Gradientenabstieg (wie in Notebook 05 mit eta = 0.001)
x_gd, path_gd = gradient_descent(rosenbrock_grad, x0_opt, eta=0.001, max_iter=20000)

# 2. Newton-Verfahren
x_newton, path_newton = newton_optimizer(rosenbrock_grad, rosenbrock_hess, x0_opt)

print(f"Startpunkt: {x0_opt}, Minimum: (1.0, 1.0)\n")
print(f"Gradientenabstieg (eta=0.001): {len(path_gd)-1:6d} Schritte -> x = ({x_gd[0]:.6f}, {x_gd[1]:.6f}), f(x) = {rosenbrock(*x_gd):.6e}")
print(f"Newton-Verfahren:              {len(path_newton)-1:6d} Schritte -> x = ({x_newton[0]:.6f}, {x_newton[1]:.6f}), f(x) = {rosenbrock(*x_newton):.6e}")

### Iterationsschritte des Newton-Verfahrens im Detail:

In [ ]:
print(f"{'k':>2} | {'x_k':>14} | {'y_k':>14} | {'f(x_k)':>14} | {'||grad f||':>14}")
print("-" * 65)
for k, p in enumerate(path_newton):
    g_k = rosenbrock_grad(*p)
    print(f"{k:2d} | {p[0]:14.8f} | {p[1]:14.8f} | {rosenbrock(*p):14.6e} | {np.linalg.norm(g_k):14.4e}")

## Visualisierung: Konvergenz und Pfade im Höhenlinienbild

In [ ]:
f_gd = [rosenbrock(*p) for p in path_gd]
f_newton = [rosenbrock(*p) for p in path_newton]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Konvergenzkurven
axes[0].plot(f_gd, 'b-', label=f'Gradientenabstieg ({len(path_gd)-1} Schritte)')
axes[0].plot(f_newton, 'r.-', linewidth=2, markersize=8, label=f'Newton-Verfahren ({len(path_newton)-1} Schritte)')
axes[0].set_yscale('log')
axes[0].set_xlim(0, max(len(path_newton) + 5, 20))
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('$f(x, y)$ (logarithmische Skala)')
axes[0].set_title('Konvergenzverlauf (erste 20 Schritte)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Höhenlinienbild mit Pfaden
xs = np.linspace(-1.5, 1.5, 400)
ys = np.linspace(-0.5, 1.5, 400)
X, Y = np.meshgrid(xs, ys)
Z = rosenbrock(X, Y)

axes[1].contour(X, Y, Z, levels=np.logspace(-1, 3.5, 20), cmap='viridis', alpha=0.7)
axes[1].plot(path_gd[:, 0], path_gd[:, 1], 'b.-', markersize=2, linewidth=0.8, alpha=0.7, label='Gradientenabstieg')
axes[1].plot(path_newton[:, 0], path_newton[:, 1], 'r.-', markersize=6, linewidth=2, label='Newton-Verfahren')
axes[1].plot(1, 1, 'k*', markersize=12, label='Minimum $(1, 1)$')
axes[1].plot(*x0_opt, 'go', markersize=8, label=r'Startpunkt $(-1{,}2,\ 1{,}0)$')

axes[1].set_xlim(-1.5, 1.5)
axes[1].set_ylim(-0.5, 1.5)
axes[1].set_xlabel('$x$')
axes[1].set_ylabel('$y$')
axes[1].set_title('Pfade auf der Rosenbrock-Funktion')
axes[1].legend(loc='upper left', fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
